# 15 · Synthesize All Neighbourhoods

Generate synthetic populations for **every** neighbourhood in Gothenburg, produce per-area error reports, and a consolidated summary.

⏱️ This notebook processes all 96 areas and takes several minutes to run.

In [1]:
import logging
import os
import json
import traceback
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd

from gbgsynth import GbgSynth

logging.basicConfig(level=logging.WARNING)

### Configuration

In [2]:
YEAR = 2023
OUTPUT_DIR = Path.cwd().parent / "output"
POP_DIR = OUTPUT_DIR / "populations"
REPORT_DIR = OUTPUT_DIR / "reports"

POP_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

# Areas to skip (zero population or no geographic area)
SKIP_AREAS = {"199"}  # Ospecificerat

# Areas with known data quality warnings (will still be processed)
WARN_AREAS = {
    "108": "Annedal has unusual household/population ratio — results may be unreliable",
    "516": "Högsbo is primarily industrial — very small residential population",
    "707": "Arendal is primarily industrial — very small residential population",
}

print(f"Year:   {YEAR}")
print(f"Output: {OUTPUT_DIR}")

Year:   2023
Output: /Users/ssanjay/GitHub/GbgSynth/output


### Discover all areas

In [3]:
city = GbgSynth(year=YEAR)
all_areas = city.get_all_areas()
print(f"Found {len(all_areas)} neighbourhoods")

Found 96 neighbourhoods


### Helper — generate a per-area error report

In [ ]:
def generate_error_report(area, comparisons, execution_time, error=None):
    """Build a comprehensive text error report for one neighbourhood."""
    lines = []
    ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    lines.append("=" * 80)
    lines.append("SYNTHETIC POPULATION ERROR REPORT")
    lines.append("=" * 80)

    if area:
        lines.append(f"Neighbourhood: {area.area_name}")
        lines.append(f"Area Code:     {area.area_code}")
        lines.append(f"Year:          {area.year}")
    else:
        lines.append("Neighbourhood: GENERATION FAILED")

    lines.append(f"Generated:     {ts}")
    lines.append(f"Execution Time: {execution_time:.2f}s")
    lines.append("")

    if error:
        lines += ["!" * 80, "GENERATION ERROR", "!" * 80, error, "!" * 80, ""]
        return "\n".join(lines)

    # Summary statistics
    stats = area.get_summary_statistics()
    lines.append("-" * 80)
    lines.append("SUMMARY STATISTICS")
    lines.append("-" * 80)
    for label, key, fmt in [
        ("Total Population",  "total_population",       ",d"),
        ("Total Households",  "total_households",        ",d"),
        ("Average HH Size",  "avg_household_size",       ".2f"),
        ("Number of Children","num_children",             ",d"),
        ("Number of Adults",  "num_adults",              ",d"),
        ("Couple Households", "num_couples",              ",d"),
        ("Single-Parent HH",  "num_single_parent",       ",d"),
        ("Single-Person HH",  "num_single_person",       ",d"),
        ("Average Income",    "avg_income",               ",.0f"),
        ("Total Cars",        "total_cars",               ",d"),
        ("Småhus",            "hustyp_smahus",            ",d"),
        ("Flerbostadshus",    "hustyp_flerbostadshus",    ",d"),
        ("Specialbostad",     "hustyp_specialbostad",     ",d"),
    ]:
        v = stats.get(key, "N/A")
        try:
            lines.append(f"  {label + ':':30s} {v:{fmt}}")
        except (ValueError, TypeError):
            lines.append(f"  {label + ':':30s} {v}")
    lines.append("")

    # Per-dimension tables
    for key, data in comparisons.items():
        if key == "overall" or not data or "comparison" not in data:
            continue
        lines.append("-" * 80)
        lines.append(f"MARGINAL COMPARISON: {data['name'].upper()}")
        lines.append("-" * 80)
        lines.append(f"{'Category':<35} {'Census':>10} {'Synth':>10} {'Diff':>10} {'Error%':>10}")
        lines.append("-" * 75)

        sorted_rows = sorted(data["comparison"], key=lambda r: abs(r["error_pct"]), reverse=True)
        total_actual, total_synth, max_abs_error_pct = 0, 0, 0

        for row in sorted_rows:
            cat = str(row["category"])[:35]
            marker = ""
            if abs(row["error_pct"]) > 20:
                marker = " ⚠️ HIGH"
            elif abs(row["error_pct"]) > 10:
                marker = " ⚡"
            lines.append(
                f"{cat:<35} {row['actual']:>10,} {row['synth']:>10,} "
                f"{row['diff']:>+10,} {row['error_pct']:>9.1f}%{marker}"
            )
            total_actual += row["actual"]
            total_synth += row["synth"]
            max_abs_error_pct = max(max_abs_error_pct, abs(row["error_pct"]))

        # Subtotals
        lines.append("-" * 75)
        is_sek = key == "median_income"
        is_info = key in ("median_income", "hh_type_children", "joint_role_age_sex")
        n = len(sorted_rows)
        if is_sek and n > 0:
            avg_a, avg_s = total_actual // n, total_synth // n
            avg_d = avg_s - avg_a
            avg_e = (avg_d / avg_a * 100) if avg_a else 0
            lines.append(f"{'AVERAGE':<35} {avg_a:>10,} {avg_s:>10,} {avg_d:>+10,} {avg_e:>9.1f}%")
        else:
            td = total_synth - total_actual
            te = (td / total_actual * 100) if total_actual else 0
            lines.append(f"{'TOTAL':<35} {total_actual:>10,} {total_synth:>10,} {td:>+10,} {te:>9.1f}%")
        if is_info:
            lines.append("  (informational — excluded from MAPE grade)")

        # Quality assessment
        if max_abs_error_pct > 25:
            quality = "POOR - Significant discrepancies detected"
        elif max_abs_error_pct > 15:
            quality = "FAIR - Some notable discrepancies"
        elif max_abs_error_pct > 5:
            quality = "GOOD - Minor discrepancies"
        else:
            quality = "EXCELLENT - Very close match"
        lines.append(f"Quality Assessment: {quality}")
        lines.append("")

    # Overall fit with Voas & Williamson metrics
    if "overall" in comparisons:
        ov = comparisons["overall"]
        lines.append("=" * 80)
        lines.append("OVERALL FIT STATISTICS")
        lines.append("=" * 80)
        lines.append(f"  Total Census Population:     {ov['total_actual']:>15,}")
        lines.append(f"  Total Synthetic Population:  {ov['total_synth']:>15,}")
        lines.append(f"  Population Difference:       {ov['total_synth'] - ov['total_actual']:>+15,}")
        lines.append("")
        lines.append(f"  RMSE:        {ov['rmse']:.2f}")
        lines.append(f"  MAE:         {ov['mae']:.2f}")
        lines.append(f"  Max Error:   {ov['max_error']:,}")
        lines.append(f"  Correlation: {ov['correlation']:.4f}")
        lines.append(f"  MAPE:        {ov.get('mape', 0):.1f}%")
        lines.append(f"  WMAPE:       {ov.get('wmape', 0):.1f}%")
        lines.append("")
        lines.append("  Voas & Williamson (2001) Goodness-of-Fit (per-dimension):")
        lines.append(f"    SAE median:                {ov.get('sae_median', 0):>10.4f}  ← lower is better")
        lines.append(f"    SAE mean:                  {ov.get('sae_mean', 0):>10.4f}")
        lines.append(f"    SAE max (worst dim):       {ov.get('sae_max', 0):>10.4f}")
        lines.append(f"    X² p-value (worst dim):    {ov.get('chi2_p_min', 0):>10.4f}  ← higher is better")
        lines.append(f"    Z² p-value (worst dim):    {ov.get('z2_p_min', 0):>10.4f}")
        dm = ov.get("dim_metrics", {})
        if dm:
            lines.append("")
            lines.append(f"    {'Dimension':<20} {'SAE':>8} {'X² p':>8} {'Z² p':>8}")
            lines.append("    " + "-" * 44)
            for dim_key, m in dm.items():
                lines.append(f"    {dim_key:<20} {m['sae']:>8.4f} {m['chi2_p']:>8.4f} {m['z2_p']:>8.4f}")
        lines.append("")

        # Interpretation (SAE from Voas & Williamson, 2001 — lower is better)
        sae = ov.get("sae_median", 1.0)
        lines.append(f"  Median SAE:  {sae:.4f}  (lower → better fit; 0 = perfect)")
        lines.append("")

    # Recommendations
    lines.append("=" * 80)
    lines.append("RECOMMENDATIONS")
    lines.append("=" * 80)
    recommendations = []
    for key, data in comparisons.items():
        if key == "overall" or not data or "comparison" not in data:
            continue
        for row in data["comparison"]:
            if abs(row["error_pct"]) > 20:
                recommendations.append(
                    f"- Review '{row['category']}' in {data['name']}: "
                    f"{abs(row['error_pct']):.1f}% error"
                )
    if "overall" in comparisons:
        if comparisons["overall"]["correlation"] < 0.95:
            recommendations.append(
                "- Low correlation suggests systematic mismatch. "
                "Consider using constrained IPF."
            )
        if comparisons["overall"]["rmse"] > 100:
            recommendations.append("- High RMSE indicates large absolute errors in some categories")
    if not recommendations:
        recommendations.append("- Synthesis quality is acceptable. No specific recommendations.")
    for rec in recommendations[:10]:
        lines.append(rec)
    lines.append("")
    lines.append("=" * 80)
    lines.append("END OF REPORT")
    lines.append("=" * 80)

    return "\n".join(lines)

### Synthesize every neighbourhood

Each area is synthesized, saved to CSV, and an error report is written.

In [5]:
all_results = []
successful, failed, skipped = 0, 0, 0
total = len(all_areas)

for idx, (code, name) in enumerate(all_areas.items(), 1):
    safe_name = name.replace(" ", "_").replace("/", "-")

    if code in SKIP_AREAS:
        print(f"[{idx:>3}/{total}] ⏭️  {name} — skipped")
        skipped += 1
        continue

    if code in WARN_AREAS:
        print(f"  ⚠️  Warning: {WARN_AREAS[code]}")

    result = dict(area_code=code, area_name=name, status="pending",
                  execution_time=0, error=None, stats=None, comparisons=None)
    t0 = datetime.now()

    try:
        area = city.get_area(code)
        area.generate()
        dt = (datetime.now() - t0).total_seconds()
        result["execution_time"] = dt

        stats = area.get_summary_statistics()
        comparisons = area.compare_to_marginals(print_report=False)
        result.update(stats=stats, comparisons=comparisons, status="success")

        # Save CSVs
        area.save_to_csv(str(POP_DIR / f"{code}_{safe_name}_individuals.csv"))
        area.save_households_to_csv(str(POP_DIR / f"{code}_{safe_name}_households.csv"))

        # Save error report
        report = generate_error_report(area, comparisons, dt)
        (REPORT_DIR / f"{code}_{safe_name}_error_report.txt").write_text(report, encoding="utf-8")

        successful += 1
        ov = comparisons.get("overall", {})
        corr = ov.get("correlation", 0)
        mape = ov.get("mape", 0)
        sae = ov.get("sae_median", 0)
        chi2_p = ov.get("chi2_p_min", 0)
        print(f"[{idx:>3}/{total}] ✅ {name:<25s}  pop={stats['total_population']:>5,}  "
              f"SAE={sae:.4f}  MAPE={mape:>5.1f}%  X²p={chi2_p:.4f}  ({dt:.1f}s)")

    except Exception as e:
        dt = (datetime.now() - t0).total_seconds()
        result.update(execution_time=dt, status="failed",
                      error=f"{type(e).__name__}: {e}\n{traceback.format_exc()}")
        report = generate_error_report(None, {}, dt, result["error"])
        (REPORT_DIR / f"{code}_{safe_name}_error_report.txt").write_text(report, encoding="utf-8")
        failed += 1
        print(f"[{idx:>3}/{total}] ❌ {name:<25s}  {e}")

    all_results.append(result)

print(f"\nDone: {successful} succeeded, {failed} failed, {skipped} skipped")

[  1/96] ✅ 101 Kungsladugård          pop=10,815  SAE=0.0014  MAPE=  1.4%  X²p=0.4223  (2.1s)


[  2/96] ✅ 102 Sanna                  pop=3,346  SAE=0.0161  MAPE=  2.5%  X²p=0.1690  (0.6s)
[  3/96] ✅ 103 Majorna                pop=10,955  SAE=0.0035  MAPE=  2.8%  X²p=0.6247  (1.4s)


[  4/96] ✅ 104 Stigberget             pop=7,781  SAE=0.0051  MAPE=  2.6%  X²p=0.1153  (12.5s)


[  5/96] ✅ 105 Masthugget             pop=11,431  SAE=0.0026  MAPE=  1.1%  X²p=0.6635  (1.5s)
[  6/96] ✅ 106 Änggården              pop=1,473  SAE=0.0138  MAPE=  2.1%  X²p=0.8805  (0.5s)


[  7/96] ✅ 107 Haga                   pop=3,808  SAE=0.0049  MAPE=  1.5%  X²p=0.9395  (0.3s)
  ⚠️  Warning: Annedal has unusual household/population ratio — results may be unreliable
[  8/96] ✅ 108 Annedal                pop=4,266  SAE=0.0033  MAPE=  2.5%  X²p=0.8455  (0.4s)


...

[ 88/96] ✅ 701 Norra Biskopsgården    pop=5,347  SAE=0.0180  MAPE=  3.6%  X²p=0.0680  (1.0s)


[ 89/96] ✅ 702 Länsmansgården         pop=5,891  SAE=0.0069  MAPE=  2.6%  X²p=0.6032  (12.3s)
[ 90/96] ✅ 703 Svartedalen            pop=4,591  SAE=0.0033  MAPE=  2.5%  X²p=0.3079  (0.9s)
[ 91/96] ✅ 704 Hjuvik                 pop=7,426  SAE=0.0016  MAPE=  1.3%  X²p=0.3082  (2.3s)


[ 92/96] ✅ 705 Nolered                pop=10,999  SAE=0.0024  MAPE=  1.5%  X²p=0.3156  (15.5s)
[ 93/96] ✅ 706 Björlanda              pop=8,956  SAE=0.0019  MAPE=  2.0%  X²p=0.3847  (3.0s)
  ⚠️  Warning: Arendal is primarily industrial — very small residential population
[ 94/96] ❌ 707 Arendal                Total of weights must be greater than zero


[ 95/96] ✅ 708 Södra Biskopsgården    pop=8,373  SAE=0.0062  MAPE=  2.0%  X²p=0.7368  (1.5s)


[ 96/96] ✅ 709 Jättesten              pop=7,746  SAE=0.0024  MAPE=  2.5%  X²p=0.7622  (12.6s)

Done: 95 succeeded, 1 failed, 0 skipped


### Consolidated summary report

In [ ]:
rows = []
for r in all_results:
    row = dict(area_code=r["area_code"], area_name=r["area_name"],
               status=r["status"], time_sec=r["execution_time"])
    if r["stats"]:
        row.update(population=r["stats"]["total_population"],
                   households=r["stats"]["total_households"],
                   avg_hh_size=r["stats"]["avg_household_size"],
                   num_children=r["stats"].get("num_children"),
                   num_adults=r["stats"].get("num_adults"),
                   cars=r["stats"]["total_cars"])
    if r["comparisons"] and "overall" in r["comparisons"]:
        ov = r["comparisons"]["overall"]
        row.update(census_population=ov.get("total_actual"),
                   synth_population=ov.get("total_synth"),
                   rmse=ov.get("rmse"), mae=ov.get("mae"),
                   correlation=ov.get("correlation"),
                   max_error=ov.get("max_error"),
                   mape=ov.get("mape"), wmape=ov.get("wmape"),
                   sae_median=ov.get("sae_median"),
                   sae_mean=ov.get("sae_mean"),
                   sae_max=ov.get("sae_max"),
                   chi2_p_min=ov.get("chi2_p_min"),
                   z2_p_min=ov.get("z2_p_min"))

    # Per-dimension error summaries
    if r["comparisons"]:
        for dim_key, data in r["comparisons"].items():
            if dim_key == "overall" or not data or "comparison" not in data:
                continue
            errors = [abs(c["error_pct"]) for c in data["comparison"]]
            if errors:
                dim = dim_key.replace(" ", "_").lower()
                row[f"{dim}_max_error_pct"] = max(errors)
                row[f"{dim}_mean_error_pct"] = np.mean(errors)

    rows.append(row)

summary_df = pd.DataFrame(rows)
summary_df.to_csv(OUTPUT_DIR / "summary_report.csv", index=False)
print(f"Saved summary_report.csv ({len(summary_df)} rows)")
summary_df.head(10)

Saved summary_report.csv (96 rows)


,area_code,area_name,status,time_sec,population,households,avg_hh_size,num_children,num_adults,cars,...,education_max_error_pct,education_mean_error_pct,income_source_max_error_pct,income_source_mean_error_pct,median_income_max_error_pct,median_income_mean_error_pct,hh_type_children_max_error_pct,hh_type_children_mean_error_pct,joint_role_age_sex_max_error_pct,joint_role_age_sex_mean_error_pct
0,101,101 Kungsladugård,success,2.063092,10815.0,6077.0,1.779661,1682.0,9133.0,2610.0,...,0.8,0.350,2.9,2.566667,100.0,15.500000,478.6,93.613333,0.0,0.000000
1,102,102 Sanna,success,0.627948,3346.0,2228.0,1.501795,337.0,3009.0,603.0,...,2.0,0.675,5.5,4.111111,100.0,12.850000,100.0,22.473333,27.0,4.800000
2,103,103 Majorna,success,1.397620,10955.0,6312.0,1.735583,1669.0,9286.0,2689.0,...,0.7,0.400,2.1,1.777778,100.0,11.769811,788.9,87.953333,1.0,0.183333
3,104,104 Stigberget,success,12.513153,7781.0,4222.0,1.842965,1402.0,6379.0,1589.0,...,0.2,0.075,NaN,NaN,49.1,9.000000,490.9,74.793333,12.6,2.533333
4,105,105 Masthugget,success,1.545671,11431.0,6568.0,1.740408,1637.0,9794.0,2529.0,...,0.1,0.050,3.4,2.722222,42.7,7.939583,900.0,96.793333,3.3,0.616667
5,106,106 Änggården,success,0.454150,1473.0,665.0,2.215038,254.0,1219.0,383.0,...,2.4,1.550,4.6,2.155556,100.0,17.905000,100.0,42.106667,0.0,0.000000
6,107,107 Haga,success,0.311084,3808.0,2041.0,1.865752,569.0,3239.0,740.0,...,0.7,0.350,4.3,2.211111,39.2,9.835556,300.0,53.500000,6.8,1.383333
7,108,108 Annedal,success,0.364799,4266.0,2598.0,1.642032,453.0,3813.0,928.0,...,0.4,0.275,3.4,1.622222,100.0,13.895918,100.0,28.140000,0.0,0.000000
8,109,109 Olivedal,success,1.437736,11075.0,6247.0,1.772851,1319.0,9756.0,2605.0,...,0.2,0.100,2.5,2.000000,41.5,7.058824,625.0,82.513333,0.0,0.000000
9,110,110 Krokslätt,success,14.867588,16399.0,10146.0,1.616302,1599.0,14800.0,3132.0,...,0.3,0.200,5.0,4.344444,100.0,10.766667,266.7,74.553333,0.3,0.083333


### Detailed per-category comparison CSV

In [7]:
detail_rows = []
for r in all_results:
    if not r["comparisons"]:
        continue
    for dim_key, data in r["comparisons"].items():
        if dim_key == "overall" or not data or "comparison" not in data:
            continue
        for comp in data["comparison"]:
            detail_rows.append(dict(
                area_code=r["area_code"], area_name=r["area_name"],
                dimension=data["name"], category=comp["category"],
                census_count=comp["actual"], synth_count=comp["synth"],
                difference=comp["diff"], error_pct=comp["error_pct"]
            ))

detail_df = pd.DataFrame(detail_rows)
detail_df.to_csv(OUTPUT_DIR / "detailed_comparisons.csv", index=False)
print(f"Saved detailed_comparisons.csv ({len(detail_df)} rows)")

Saved detailed_comparisons.csv (10627 rows)


### JSON results

In [8]:
json_out = []
for r in all_results:
    entry = dict(area_code=r["area_code"], area_name=r["area_name"],
                 status=r["status"], execution_time=r["execution_time"],
                 error=r["error"])
    if r["stats"]:
        entry["stats"] = r["stats"]
    if r["comparisons"] and "overall" in r["comparisons"]:
        ov = r["comparisons"]["overall"]
        # Exclude dim_metrics (nested, verbose) from JSON for cleanliness
        entry["overall_fit"] = {k: v for k, v in ov.items() if k != "dim_metrics"}
        # Top-level shortcuts
        entry["sae_median"] = ov.get("sae_median")
        entry["mape"] = ov.get("mape")
        entry["chi2_p_min"] = ov.get("chi2_p_min")
        entry["z2_p_min"] = ov.get("z2_p_min")
    json_out.append(entry)

(OUTPUT_DIR / "results.json").write_text(
    json.dumps(json_out, indent=2, ensure_ascii=False), encoding="utf-8"
)
print(f"Saved results.json ({len(json_out)} entries)")

Saved results.json (96 entries)


### Overall statistics

In [ ]:
ok = [r for r in all_results if r["status"] == "success"]

total_pop = sum(r["stats"]["total_population"] for r in ok)
total_hh  = sum(r["stats"]["total_households"] for r in ok)

corrs = [r["comparisons"]["overall"]["correlation"]
         for r in ok if r["comparisons"] and "overall" in r["comparisons"]]
mapes = [r["comparisons"]["overall"].get("mape", 0)
         for r in ok if r["comparisons"] and "overall" in r["comparisons"]]
rmses = [r["comparisons"]["overall"].get("rmse", 0)
         for r in ok if r["comparisons"] and "overall" in r["comparisons"]]
saes = [r["comparisons"]["overall"].get("sae_median", 0)
        for r in ok if r["comparisons"] and "overall" in r["comparisons"]]

print("=" * 60)
print("SYNTHESIS COMPLETE")
print("=" * 60)
print(f"  Successful areas:    {len(ok)}")
print(f"  Skipped:             {skipped}")
print(f"  Failed:              {failed}")
print(f"  Total population:    {total_pop:,}")
print(f"  Total households:    {total_hh:,}")
print(f"  Average RMSE:        {np.mean(rmses):.2f}")
print(f"  Average MAPE:        {np.mean(mapes):.1f}%")
print(f"  Average correlation: {np.mean(corrs):.4f}")
print(f"  Average SAE:         {np.mean(saes):.4f}")
print(f"  Median SAE:          {np.median(saes):.4f}")
print()
print("SAE distribution (Voas & Williamson, 2001 — lower is better):")
for label, lo, hi in [("SAE ≤ 0.005", 0, 0.0051), ("SAE 0.005–0.01", 0.0051, 0.0101),
                       ("SAE 0.01–0.02", 0.0101, 0.0201), ("SAE 0.02–0.05", 0.0201, 0.0501),
                       ("SAE > 0.05", 0.0501, 999)]:
    n = sum(1 for s in saes if lo <= s < hi)
    pct = 100 * n / len(saes) if saes else 0
    print(f"  {label:20s}: {n:3d} areas ({pct:.0f}%)")

SYNTHESIS COMPLETE
  Successful areas:    95
  Skipped:             0
  Failed:              1
  Total population:    600,541
  Total households:    291,036
  Average RMSE:        18.01
  Average MAPE:        2.5%
  Average correlation: 0.9997
  Average SAE:         0.0072
  Median SAE:          0.0047

Quality grades (by SAE — Voas & Williamson, GenSynthPop benchmark):
  A (≤0.005):  53 areas (56%)
  B (0.005–0.01):  19 areas (20%)
  C (0.01–0.02):  18 areas (19%)
  D (0.02–0.05):   5 areas (5%)
  F (>0.05):   0 areas (0%)
